In [1]:
# 1) Import libraries

import pandas as pd
import numpy as np
from IPython.display import display

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# 2) Load raw network traffic file

data_path = "../data/raw/Week_filtered.csv"

df = pd.read_csv(data_path)

print("Raw data loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Raw data loaded successfully.
Shape: (543734, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,115550790,31,33,318,38380,318,0,10.258065,57.114486,...,32,29534.636364,20922.826407,92617,22941,1.002161e+07,8209.967001,10025078,9998039,BENIGN
1,443,5292756,6,4,611,168,517,0,101.833333,204.305082,...,20,0.000000,0.000000,0,0,0.000000e+00,0.000000,0,0,BENIGN
2,21,3,2,0,14,0,14,0,7.000000,9.899495,...,32,0.000000,0.000000,0,0,0.000000e+00,0.000000,0,0,FTP-Patator
3,54045,41,1,1,2,6,2,2,2.000000,0.000000,...,24,0.000000,0.000000,0,0,0.000000e+00,0.000000,0,0,PortScan
4,80,1023,2,0,0,0,0,0,0.000000,0.000000,...,32,0.000000,0.000000,0,0,0.000000e+00,0.000000,0,0,DoS Hulk


In [3]:
# 3) Clean column names

df.columns = df.columns.str.strip()

print("Column names cleaned.")
print("First 20 columns:")
print(df.columns[:20].tolist())

Column names cleaned.
First 20 columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min']


In [ ]:
# 4) Define signature-based detection rules

def signature_detection(row):
    alerts = []

    def get_value(col, default=0):
        if col in row.index:
            value = row[col]
            if pd.isna(value):
                return default
            return value
        return default

   
    # Rule 1: Possible DoS / DDoS pattern
  
    if (
        get_value("Destination Port", 0) in [80, 443]
        and get_value("Flow Packets/s", 0) > 10000
    ):
        alerts.append("Possible DoS/DDoS pattern detected")

   
    # Rule 2: SYN flood pattern
   
    if get_value("SYN Flag Count", 0) > 20:
        alerts.append("Possible SYN flood detected")

    
    # Rule 3: Port scan pattern
   
    if (
        get_value("Flow Duration", 0) < 50000
        and get_value("Total Fwd Packets", 0) < 5
        and get_value("Total Backward Packets", 0) < 5
    ):
        alerts.append("Possible port scanning activity")

    
    # Rule 4: Suspiciously high byte rate
  
    if get_value("Flow Bytes/s", 0) > 1000000:
        alerts.append("Suspiciously high byte rate")


    # Rule 5: Suspicious FTP brute-force activity
    
    if (
        get_value("Destination Port", 0) == 21
        and get_value("Flow Packets/s", 0) > 500
    ):
        alerts.append("Possible FTP brute-force activity")

    if alerts:
        return "INTRUSION", "; ".join(alerts)

    return "BENIGN", "No signature rule matched"

In [5]:
# 5) Apply signature-based detection to all rows

signature_results = []

for _, row in df.iterrows():
    status, reason = signature_detection(row)

    signature_results.append({
        "Signature_Status": status,
        "Signature_Reason": reason
    })

signature_df = pd.DataFrame(signature_results)

print("Signature detection completed.")
display(signature_df.head(20))

Signature detection completed.


,Signature_Status,Signature_Reason
0,BENIGN,No signature rule matched
1,BENIGN,No signature rule matched
2,INTRUSION,Possible port scanning activity; Suspiciously ...
3,INTRUSION,Possible port scanning activity
4,INTRUSION,Possible port scanning activity
5,INTRUSION,Possible port scanning activity
6,INTRUSION,Possible port scanning activity
7,BENIGN,No signature rule matched
8,BENIGN,No signature rule matched
9,BENIGN,No signature rule matched


In [6]:
# 6) Combine original data with signature results

results_df = pd.concat([df.reset_index(drop=True), signature_df], axis=1)

print("Combined results preview:")
display(results_df.head(20))

Combined results preview:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Signature_Status,Signature_Reason
0,80,115550790,31,33,318,38380,318,0,10.258065,57.114486,...,20922.826407,92617,22941,1.002161e+07,8209.967001,10025078,9998039,BENIGN,BENIGN,No signature rule matched
1,443,5292756,6,4,611,168,517,0,101.833333,204.305082,...,0.000000,0,0,0.000000e+00,0.000000,0,0,BENIGN,BENIGN,No signature rule matched
2,21,3,2,0,14,0,14,0,7.000000,9.899495,...,0.000000,0,0,0.000000e+00,0.000000,0,0,FTP-Patator,INTRUSION,Possible port scanning activity; Suspiciously ...
3,54045,41,1,1,2,6,2,2,2.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000,0,0,PortScan,INTRUSION,Possible port scanning activity
4,80,1023,2,0,0,0,0,0,0.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000,0,0,DoS Hulk,INTRUSION,Possible port scanning activity
5,53,48119,2,2,60,92,30,30,30.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000,0,0,BENIGN,INTRUSION,Possible port scanning activity
6,63286,60,1,1,6,6,6,6,6.000000,0.000000,...,0.000000,0,0,0.000000e+00,0.000000,0,0,BENIGN,INTRUSION,Possible port scanning activity
7,80,71548585,8,5,56,11601,20,0,7.000000,5.656854,...,0.000000,1040891,1040891,7.030000e+07,0.000000,70300000,70300000,DDoS,BENIGN,No signature rule matched
8,443,588453,15,10,974,7387,469,0,64.933333,134.265976,...,0.000000,0,0,0.000000e+00,0.000000,0,0,BENIGN,BENIGN,No signature rule matched
9,80,57772,3,6,26,11607,20,0,8.666667,10.263203,...,0.000000,0,0,0.000000e+00,0.000000,0,0,DDoS,BENIGN,No signature rule matched


In [7]:
# 7) Summary counts

print("Signature Status Summary:")
print(results_df["Signature_Status"].value_counts())

print("\nTop Signature Reasons:")
print(results_df["Signature_Reason"].value_counts().head(10))

Signature Status Summary:
Signature_Status
BENIGN       287731
INTRUSION    256003
Name: count, dtype: int64

Top Signature Reasons:
Signature_Reason
No signature rule matched                                                                           287731
Possible port scanning activity                                                                     183966
Possible port scanning activity; Suspiciously high byte rate                                         35476
Possible DoS/DDoS pattern detected; Possible port scanning activity                                  27450
Possible DoS/DDoS pattern detected; Possible port scanning activity; Suspiciously high byte rate      3226
Possible port scanning activity; Possible FTP brute-force activity                                    3109
Suspiciously high byte rate                                                                           1494
Possible port scanning activity; Suspiciously high byte rate; Possible FTP brute-force activity      

In [8]:
# 8) Show only detected attacks

detected_attacks_df = results_df[results_df["Signature_Status"] == "INTRUSION"].copy()

print("Detected signature-based attacks:")
display(detected_attacks_df.head(20))

print("Total signature-based attack rows:", len(detected_attacks_df))

Detected signature-based attacks:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Signature_Status,Signature_Reason
2,21,3,2,0,14,0,14,0,7.000000,9.899495,...,0.0,0,0,0.0,0.0,0,0,FTP-Patator,INTRUSION,Possible port scanning activity; Suspiciously ...
3,54045,41,1,1,2,6,2,2,2.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,PortScan,INTRUSION,Possible port scanning activity
4,80,1023,2,0,0,0,0,0,0.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,DoS Hulk,INTRUSION,Possible port scanning activity
5,53,48119,2,2,60,92,30,30,30.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,BENIGN,INTRUSION,Possible port scanning activity
6,63286,60,1,1,6,6,6,6,6.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,BENIGN,INTRUSION,Possible port scanning activity
10,80,26830,3,4,26,11601,20,0,8.666667,10.263203,...,0.0,0,0,0.0,0.0,0,0,DDoS,INTRUSION,Possible port scanning activity
11,1050,71,1,1,2,6,2,2,2.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,PortScan,INTRUSION,Possible port scanning activity
12,80,3,2,0,0,0,0,0,0.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,DoS Hulk,INTRUSION,Possible DoS/DDoS pattern detected; Possible p...
16,443,207,3,1,18,0,6,6,6.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,BENIGN,INTRUSION,Possible DoS/DDoS pattern detected; Possible p...
18,1046,114,1,1,2,6,2,2,2.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,PortScan,INTRUSION,Possible port scanning activity


Total signature-based attack rows: 256003


In [9]:
# 9) Save signature detection output

output_path = "../data/predictions/signature_only_predictions.csv"

results_df.to_csv(output_path, index=False)

print(f"Signature-based results saved to: {output_path}")

Signature-based results saved to: ../data/predictions/signature_only_predictions.csv
